# 3장 짝지은 자료, 세 군 이상, 비율의 비교 (R과 파이썬으로 하는 통계분석)

보건통계학 기초 3–5장의 예로 대응표본 t 검정과 부호순위 검정, 분산분석과 Welch 분산분석, 사후분석, Kruskal-Wallis 검정과 Dunn 검정, 추세 검정, 카이제곱 검정과 위험차·상대위험도·오즈비, McNemar 검정, Fisher의 정확한 검정, 선형 대 선형 결합 검정을 R로 계산합니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요. 패키지가 없으면 먼저 install.packages(c("tidyr", "coin", "car", "multcomp", "rstatix", "DescTools"))를 실행합니다.


## a. 짝지은 두 값의 평균 비교

### 셀 1. 대응표본 t 검정

In [ ]:
before <- c(131, 146, 144, 154, 164, 159, 155, 164, 149, 161, 155, 142)   # 교육 전
after  <- c(126, 144, 131, 147, 162, 150, 154, 147, 142, 154, 139, 144)   # 교육 3개월 뒤
t.test(after, before, paired = TRUE)          # 후 − 전

### 셀 2. 짝을 무시하면

In [ ]:
d <- after - before
c(mean = mean(d), sd = sd(d), n = length(d))
t.test(after, before, var.equal = TRUE)$p.value   # 잘못된 분석: 두 독립 군처럼

### 셀 3. 측정마다 한 줄인 자료를 환자당 한 줄로

In [ ]:
library(tidyr)
long <- data.frame(id = c(1, 1, 2, 2, 3),
                   time = c("base", "m3", "base", "m3", "base"),
                   sbp = c(131, 126, 146, 144, 144))      # 측정마다 한 줄, 3번은 3개월 값 없음
wide <- pivot_wider(long, names_from = time, values_from = sbp)
wide
sum(complete.cases(wide))                              # 짝이 맞는 환자 수

## b. 부호순위 검정과 부호 검정

### 셀 4. 부호순위 검정의 정규근사

In [ ]:
before <- c(12, 10, 14, 9, 11, 13, 10, 12, 15, 11, 9, 13, 10, 12)   # 약물 정리 전 복용 약물 수
after  <- c( 9,  9, 11, 9, 10, 10,  8, 12, 11,  9, 10, 10, 12,  9)   # 3개월 뒤
d <- after - before
table(sign(d))                                                   # 감소, 변화 없음, 증가
wilcox.test(after, before, paired = TRUE, exact = FALSE, correct = FALSE)$p.value
wilcox.test(after, before, paired = TRUE, exact = FALSE)$p.value   # 연속성 보정

### 셀 5. 정확 p, 부호 검정, t 검정

In [ ]:
suppressPackageStartupMessages(library(coin))
dd <- data.frame(after, before)
pvalue(wilcoxsign_test(after ~ before, data = dd, zero.method = "Wilcoxon",
                       distribution = "exact"))                  # 0을 빼고 정확 p
binom.test(sum(d > 0), sum(d != 0))$p.value                      # 부호 검정
t.test(after, before, paired = TRUE)$p.value                     # 대응표본 t 검정

### 셀 6. Hodges–Lehmann 추정치

In [ ]:
d0 <- d[d != 0]                                  # 0을 뺀 차이 12개
walsh <- outer(d0, d0, "+") / 2                   # 두 차이의 평균
median(walsh[upper.tri(walsh, diag = TRUE)])      # Walsh 평균 78개의 중앙값

## c. 분산분석과 Welch 분산분석

### 셀 7. 분산분석표

In [ ]:
pdc <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/ch04_pdc.csv")
pdc$arm <- factor(pdc$arm, levels = c("usual", "brief", "intensive"))   # 군의 순서와 기준군
tapply(pdc$pdc, pdc$arm, mean)
fit <- aov(pdc ~ arm, data = pdc)
summary(fit)

### 셀 8. Welch 분산분석

In [ ]:
oneway.test(pdc ~ arm, data = pdc)                        # 기본값이 Welch
oneway.test(pdc ~ arm, data = pdc, var.equal = TRUE)$p.value

### 셀 9. 등분산과 잔차의 정규성

In [ ]:
suppressPackageStartupMessages(library(car))
leveneTest(pdc ~ arm, data = pdc)                         # 기본값: 중앙값 기준
leveneTest(pdc ~ arm, data = pdc, center = mean)$`Pr(>F)`[1]
shapiro.test(residuals(fit))

### 셀 10. 분산분석을 회귀로

In [ ]:
lmfit <- lm(pdc ~ arm, data = pdc)                        # 기준군은 factor의 첫 수준(usual)
round(summary(lmfit)$coefficients, 4)
summary(lmfit)$r.squared

## d. 사후분석

### 셀 11. Tukey 방법

In [ ]:
TukeyHSD(fit)

### 셀 12. Dunnett 방법

In [ ]:
suppressPackageStartupMessages(library(multcomp))
set.seed(1)
dn <- glht(fit, linfct = mcp(arm = "Dunnett"))          # 첫 수준(usual)이 대조군
summary(dn)

### 셀 13. Holm과 Games-Howell

In [ ]:
pairwise.t.test(pdc$pdc, pdc$arm, p.adjust.method = "holm")   # 군내 평균제곱으로 t
suppressPackageStartupMessages(library(rstatix))
games_howell_test(pdc, pdc ~ arm)[, c("group1", "group2", "estimate", "p.adj")]

### 셀 14. 미리 정한 대비 하나

In [ ]:
ct <- glht(fit, linfct = mcp(arm = c("brief + intensive - 2 * usual = 0")))
s <- summary(ct)
c(diff = unname(coef(ct)) / 2, t = unname(s$test$tstat), p = unname(s$test$pvalues))

## e. 순위로 비교하기와 추세

### 셀 15. 군당 5명의 Kruskal-Wallis 검정

In [ ]:
tiny <- data.frame(arm = factor(rep(c("usual", "brief", "intensive"), each = 5),
                                levels = c("usual", "brief", "intensive")),
                   pdc = c(58, 63, 67, 69, 73,  62, 68, 71, 77, 82,  70, 74, 79, 81, 86))
kruskal.test(pdc ~ arm, data = tiny)
set.seed(2026)
pvalue(coin::kruskal_test(pdc ~ arm, data = tiny,
                          distribution = approximate(nresample = 100000)))   # 순열검정 p

### 셀 16. Kruskal-Wallis 검정과 Dunn 검정

In [ ]:
kruskal.test(pdc ~ arm, data = pdc)
kruskal.test(knowledge ~ arm, data = pdc)$statistic      # 동점이 많은 지식 점수
dunn_test(pdc, pdc ~ arm, p.adjust.method = "holm")[, c("group1", "group2", "statistic", "p", "p.adj")]

### 셀 17. 순서가 있는 세 군의 추세

In [ ]:
suppressPackageStartupMessages(library(DescTools))
suppressWarnings(JonckheereTerpstraTest(pdc ~ arm, data = pdc))   # 순서는 factor 수준 순서, 동점이 있어 정규근사
pdc$score <- as.numeric(pdc$arm)                          # usual 1, brief 2, intensive 3
round(confint(lm(pdc ~ score, data = pdc))["score", ], 2)
coef(summary(lm(pdc ~ score, data = pdc)))["score", ]

## f. 카이제곱 검정과 효과 지표

### 셀 18. 2 × 2 표의 카이제곱 검정

In [ ]:
tab <- matrix(c(48, 352,
                102, 498), nrow = 2, byrow = TRUE,
              dimnames = list(drug = c("A", "B"), admit = c("yes", "no")))
ct <- chisq.test(tab, correct = FALSE)
ct
ct$expected
chisq.test(tab)$p.value                      # 기본값: Yates 보정

### 셀 19. 위험차, 상대위험도, 오즈비

In [ ]:
p1 <- 48 / 400; p2 <- 102 / 600
rd <- p1 - p2; se_rd <- sqrt(p1 * (1 - p1) / 400 + p2 * (1 - p2) / 600)
rr <- p1 / p2; se_lrr <- sqrt(1/48 - 1/400 + 1/102 - 1/600)
or <- (48 * 498) / (352 * 102); se_lor <- sqrt(1/48 + 1/352 + 1/102 + 1/498)
round(rbind(RD = rd + c(0, -1.96, 1.96) * se_rd,
            RR = exp(log(rr) + c(0, -1.96, 1.96) * se_lrr),
            OR = exp(log(or) + c(0, -1.96, 1.96) * se_lor)), 3)

### 셀 20. 세 군의 비율 비교

In [ ]:
hypo <- matrix(c(12, 288, 15, 285, 33, 267), ncol = 2, byrow = TRUE,
               dimnames = list(class = c("SGLT2", "DPP4", "SU"), hypo = c("yes", "no")))
ch <- chisq.test(hypo)
c(X2 = unname(ch$statistic), df = unname(ch$parameter), p = ch$p.value)
round(ch$stdres[, "yes"], 2)                                  # 수정 표준화 잔차
pairwise.prop.test(hypo, correct = FALSE, p.adjust.method = "bonferroni")$p.value

### 셀 21. 같은 환자의 전후 비율

In [ ]:
pre_post <- matrix(c(47, 7,
                     25, 41), nrow = 2, byrow = TRUE,
                   dimnames = list(before = c("adh", "non"), after = c("adh", "non")))
mcnemar.test(pre_post)$p.value                     # 기본값: 연속성 보정
binom.test(7, 7 + 25)$p.value                      # 바뀐 32명만 보는 정확 검정
chisq.test(rbind(colSums(pre_post), rowSums(pre_post)), correct = FALSE)$p.value   # 잘못된 분석

## g. Fisher의 정확한 검정

### 셀 22. Fisher의 정확한 검정

In [ ]:
tabF <- matrix(c(1, 14,
                 6,  8), nrow = 2, byrow = TRUE,
               dimnames = list(drug = c("X", "Y"), liver = c("yes", "no")))
fisher.test(tabF)
fisher.test(tabF, alternative = "less")$p.value
(1 * 8) / (14 * 6)                                  # 표본 오즈비

### 셀 23. 보수성을 줄이는 두 방법

In [ ]:
p_obs <- dhyper(1, 15, 14, 7)                       # 관측된 표의 확률
c(midp = fisher.test(tabF)$p.value - p_obs / 2,
  N1 = pchisq(chisq.test(tabF, correct = FALSE)$statistic * 28 / 29, 1, lower.tail = FALSE))

## h. 비율의 추세

### 셀 24. 선형 대 선형 결합 검정과 Cochran–Armitage 검정

In [ ]:
admit <- c(30, 36, 40); n <- c(200, 300, 500)          # PDC < 40%, 40–79%, ≥ 80%
chisq.test(cbind(admit, n - admit))$statistic          # 순서를 무시한 Pearson χ²
x <- rep(rep(1:3, 2), c(admit, n - admit))              # 점수
y <- rep(c(1, 0), c(sum(admit), sum(n - admit)))        # 입원 여부
r <- cor(x, y)
c(M2 = (length(x) - 1) * r^2, p = pchisq((length(x) - 1) * r^2, 1, lower.tail = FALSE))
prop.trend.test(admit, n)                               # Cochran–Armitage (N r²)

## i. 과제 정답

### 셀 25. 과제 1 정답: 극단값 하나

In [ ]:
after2 <- after; after2[9] <- 3                      # 환자 9의 변화가 −4가 아니라 −12
c(t = t.test(after2, before, paired = TRUE)$p.value,
  signrank = pvalue(wilcoxsign_test(after2 ~ before, data = data.frame(after2, before),
                                    zero.method = "Wilcoxon", distribution = "exact")))

### 셀 26. 과제 2 정답: 99% 동시 신뢰구간

In [ ]:
TukeyHSD(aov(pdc ~ arm, data = pdc), conf.level = 0.99)$arm["intensive-usual", ]

### 셀 27. 과제 3 정답: 두 군만 뽑은 표

In [ ]:
hypo2 <- hypo[c("DPP4", "SU"), ]
c(pearson = chisq.test(hypo2, correct = FALSE)$p.value, yates = chisq.test(hypo2)$p.value,
  fisher = fisher.test(hypo2)$p.value)